# 13 - SochDB 101

This notebook is the broad front door for SochDB.

It is meant to help a new evaluator build a quick mental model of the platform before going deeper into any one workflow.

The goal here is not to cover every feature in depth. The goal is to answer a few basic questions clearly:

- What is SochDB?
- What does the embedded/local model look like?
- How do storage and retrieval fit together?
- What should I explore next?


## 1. What SochDB is

A useful first way to think about SochDB is:

- an embedded local database
- with retrieval-oriented capabilities
- aimed at AI and knowledge workflows

The most practical early product story is not "it does everything."

The better story is:

> SochDB gives you a more unified local workflow for storing data, retrieving relevant records, and building AI-oriented applications with fewer moving parts.


## 2. Setup

Recommended install:

```bash
pip install sochdb numpy
```

This notebook stays fully local and uses deterministic embeddings so it can run without external model APIs.

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
from sochdb import Database, HnswIndex


In [ ]:
DB_PATH = Path("./sochdb_101_db")
DIMENSION = 64

if DB_PATH.exists():
    shutil.rmtree(DB_PATH)


def deterministic_embedding(text: str, dim: int = DIMENSION) -> np.ndarray:
    values = []
    counter = 0
    while len(values) < dim:
        digest = hashlib.sha256(f"{text}::{counter}".encode("utf-8")).digest()
        values.extend((byte / 255.0) * 2.0 - 1.0 for byte in digest)
        counter += 1
    vector = np.array(values[:dim], dtype=np.float32)
    norm = np.linalg.norm(vector)
    return vector if norm == 0 else vector / norm


db = Database.open(str(DB_PATH))
print(f"Database path: {DB_PATH.resolve()}")


## 3. Basic database flow

At the lowest level, SochDB can be approached like a local embedded database. You open a directory-backed DB, write records, and read them back.

In [ ]:
with db.transaction() as txn:
    db.put(b"users/alice", json.dumps({"name": "Alice", "team": "platform"}).encode("utf-8"), txn.id)
    db.put(b"users/bob", json.dumps({"name": "Bob", "team": "security"}).encode("utf-8"), txn.id)

print("alice ->", db.get(b"users/alice").decode("utf-8"))
print("bob   ->", db.get(b"users/bob").decode("utf-8"))


## 4. A tiny retrieval glimpse

Now we add one small retrieval example just to show how storage and retrieval fit together.

This is intentionally small. The deeper retrieval walkthrough lives in the dedicated local retrieval notebook.

In [ ]:
documents = [
    {
        "id": 101,
        "title": "Laptop VPN Setup",
        "body": "To access internal dashboards, install the company VPN client and connect before opening private services.",
    },
    {
        "id": 102,
        "title": "Vendor Security Review",
        "body": "New vendors handling sensitive data must complete the security questionnaire and receive approval before purchase.",
    },
    {
        "id": 103,
        "title": "Incident Rollback Checklist",
        "body": "If a deployment causes errors, stop rollout, restore the previous version, and confirm service health before resuming changes.",
    },
]

vectors = []
ids = []

with db.transaction() as txn:
    for doc in documents:
        key = f"docs/{doc['id']}".encode("utf-8")
        db.put(key, json.dumps(doc).encode("utf-8"), txn.id)
        vectors.append(deterministic_embedding(f"{doc['title']}\n{doc['body']}"))
        ids.append(doc['id'])

vectors = np.vstack(vectors).astype(np.float32)
ids = np.array(ids, dtype=np.uint64)

index = HnswIndex(dimension=DIMENSION, m=16, ef_construction=100, precision="f32")
index.add(vectors, ids)

query = "How do I access internal tools securely from my laptop?"
results = index.search(deterministic_embedding(query), k=2)

rows = []
for doc_id, score in results:
    payload = db.get(f"docs/{int(doc_id)}".encode("utf-8"))
    record = json.loads(payload.decode("utf-8"))
    rows.append({"title": record["title"], "score": round(float(score), 4)})

print("query:", query)
for row in rows:
    print(row)


## 5. The platform shape

From a first-touch point of view, the easiest way to map SochDB is:

- **database layer**: store and retrieve records locally
- **retrieval layer**: vector search and retrieval-oriented workflows
- **AI workflow layer**: context building, RAG-style patterns, agent/tool-oriented paths
- **platform breadth**: SQL, graph, policy, multitenancy, admin, and other deeper capabilities

You do not need to understand every layer on day one. The best way to evaluate SochDB is usually to start with one narrow workflow and expand from there.

## 6. Why people compare it to multiple tools

For local retrieval, a common alternative is a stack like:

- SQLite for payloads
- FAISS for vectors
- custom glue code between them

That can work well, but it also gives the evaluator more pieces to manage.

SochDB is most compelling when the evaluator cares about:

- one clearer local path
- fewer moving parts
- a more unified storage + retrieval workflow


## 7. Where to go next

The best follow-up notebooks after this one are:

- `14_local_knowledge_retrieval.ipynb` for the strongest current wedge
- `0_local_knowledge_search_walkthrough.ipynb` for a local-only walkthrough
- `6_transactions_kv.ipynb` for the database and transaction side
- `10_advanced_rag.ipynb` for deeper AI/retrieval workflows

A good outcome from this notebook is not "I now know everything SochDB does."

A good outcome is:

> I understand the platform shape, and I know which workflow to evaluate next.


In [ ]:
db.close()
print("Closed database.")
